# How do you score a match between two lists of numbers?


Your music app just played you a song by a band you've never heard of, and it fits. Somewhere in a data center, two descriptions got compared and matched.

A song isn't one number. It's loud, fast, sad, bright, a dozen qualities at once, and your taste is that same kind of tangle. No single ruler works here, because the ruler that measures loud says nothing about sad.

Try answering this yourself before you run anything: how would you boil the match between two many-featured things down to one number? It's genuinely harder than it looks.


In [ ]:
import numpy as np

# Two many-featured things: a listener's taste and two candidate songs,
# each described by the same four qualities.
qualities = ["loud", "fast", "sad", "bright"]
taste = np.array([2.0, 1.0, 0.0, 3.0])
song_a = np.array([1.0, 2.5, 1.0, 4.0])
song_b = np.array([9.0, 9.0, 9.0, 9.0])

for q, t, a, b in zip(qualities, taste, song_a, song_b):
    print(f"{q:>6}: taste={t:.2f}  song_a={a:.2f}  song_b={b:.2f}")


Before you run the next cell, try the obvious move: add up each list's numbers and compare the totals. Predict which song, song_a or song_b, gets the higher total.


In [ ]:
sum_taste = taste.sum()
sum_a = song_a.sum()
sum_b = song_b.sum()
print(f"sum(taste)={sum_taste:.2f}  sum(song_a)={sum_a:.2f}  sum(song_b)={sum_b:.2f}")

# song_b is loud on every quality and matches nothing about your taste in
# particular, but plain summing rewards raw loudness over agreement.
assert sum_b > sum_a


Summing rewards volume, not agreement. Try another naive idea: count how many entries match exactly between taste and song_a. Predict the count before you run this.


In [ ]:
exact_matches = np.sum(taste == song_a)
print(f"exact matches between taste and song_a: {exact_matches}")

# Real-valued qualities almost never land on the same number twice, even
# for a song you'd genuinely like.
assert exact_matches == 0


Both naive ideas fail the same way. One lets a single loud feature drown out the rest. The other waits on a perfection nothing real ever reaches.

Here's the move that lands it: multiply the two lists entry by entry, then add everything up.


In [ ]:
a = np.array([2.0, 1.0])
b = np.array([1.0, 3.0])

score = np.sum(a * b)
print(f"a . b = {score:.2f}")

assert score == 5.0


Now meet the instrument: one probe arrow doing the asking, and four candidate arrows holding still, each candidate the same length as the others.

Before you touch the probe: swing it all the way around, one full turn. Does any candidate's score ever drop below zero? Answer that for yourself, then run the sweep below and check.


In [ ]:
candidate_angles_deg = {"c1": 0, "c2": 45, "c3": 90, "c4": 200}
candidates = {
    name: np.array([np.cos(np.radians(deg)), np.sin(np.radians(deg))])
    for name, deg in candidate_angles_deg.items()
}

probe_angles = np.linspace(0, 2 * np.pi, 360, endpoint=False)
for name, c in candidates.items():
    scores = np.array([np.dot(np.array([np.cos(t), np.sin(t)]), c) for t in probe_angles])
    print(f"{name}: min={scores.min():.2f}  max={scores.max():.2f}")

# Every candidate here is a unit-length arrow, and none of them move.
# With length held fixed, some score still dips below zero as the probe
# turns around.
assert all(
    np.array([np.dot(np.array([np.cos(t), np.sin(t)]), c) for t in probe_angles]).min() < 0
    for c in candidates.values()
)


So while every arrow held the same length, direction was all the score ever responded to. Now watch what happens when length stops being held fixed.

Below, c2 is better aligned with the probe than c1, so c2 leads while both are length 1. Before you run this: if c1 stretches to two and a half times its length, direction unchanged, will its score catch up to c2's, overtake it, or never get close? Predict one of those three before you run it.


In [ ]:
def unit(deg):
    r = np.radians(deg)
    return np.array([np.cos(r), np.sin(r)])

probe = unit(30)
c1 = unit(80)   # direction only, length 1 for now
c2 = unit(20)   # better aligned with the probe than c1

score_c1 = np.dot(probe, c1)
score_c2 = np.dot(probe, c2)
print(f"c1 score (length 1.0): {score_c1:.2f}")
print(f"c2 score (length 1.0): {score_c2:.2f}")

# c2 is better aligned with the probe, so it leads at equal length.
assert score_c2 > score_c1

c1_stretched = c1 * 2.5   # same direction, just longer
score_c1_stretched = np.dot(probe, c1_stretched)
print(f"c1 score (length 2.5): {score_c1_stretched:.2f}")

# c1's direction never changed...
assert np.allclose(c1_stretched / np.linalg.norm(c1_stretched), c1)
# ...but stretching it lets it overtake the better-aligned c2.
assert score_c1_stretched > score_c2


Look at what your hands just found. Point the probe along a candidate and the score climbs. Point it across and the score dies to zero. Point it against and the score goes negative. One multiply-and-add, and it behaves like a meter with agree, ignore, and oppose on the dial.

So say it plainly.

**The dot product is a similarity meter.**

Two lists of numbers go in. One reading of alignment comes out. We write it a . b, where a is the probe's list and b is the candidate's, the dot standing for multiply-pairwise-and-add.


In [ ]:
target = unit(0)   # a candidate pointed along the x-axis

along = unit(0)
across = unit(90)
against = unit(180)

score_along = np.dot(along, target)
score_across = np.dot(across, target)
score_against = np.dot(against, target)

print(f"along:   {score_along:.2f}")
print(f"across:  {score_across:.2f}")
print(f"against: {score_against:.2f}")

assert score_along > 0
assert abs(score_across) < 1e-9
assert score_against < 0


Rebuild the meter from memory: two arrows go in, so what two operations turn them into the score? Hold that answer while we go back to catalog scale.

Your taste is the probe. The catalog is a shelf of candidates. Run the same multiply-and-add against every song on the shelf and take the top score.


In [ ]:
rng = np.random.default_rng(0)
catalog = rng.uniform(-3, 3, size=(6, 4))
catalog_scores = catalog @ taste

for i, s in enumerate(catalog_scores):
    print(f"song {i}: score = {s:.2f}")

winner = int(np.argmax(catalog_scores))
print(f"top match: song {winner}, score {catalog_scores[winner]:.2f}")

assert catalog_scores[winner] == catalog_scores.max()


That was the whole mystery: your meter, at scale. But you saw length raise a score in the last section. The honest look asks by how much.

Point the probe exactly along c1's direction and set c1's length to 1.0. That's your baseline. Now double c1's length exactly, same direction: does the score double exactly, to the last digit, or does it just drift up? Predict a number before you run this.


In [ ]:
probe_taste = np.array([4.2, 0.0])
c1_unit = np.array([1.0, 0.0])   # pointed exactly along the probe, length 1

score_before = np.dot(probe_taste, c1_unit)
print(f"score at length 1.0: {score_before:.2f}")

c1_doubled = c1_unit * 2.0   # same direction, exactly twice as long
score_after = np.dot(probe_taste, c1_doubled)
print(f"score at length 2.0: {score_after:.2f}")

assert score_after == 2 * score_before


Double one input, double the output: the meter is linear in each arrow you feed it. That clean scaling is the crack. A loud song wins on sheer loudness, whether or not the taste underneath agrees, because size passes straight through the multiply-and-add.

Take it with you: a song and a taste became two tangles of qualities, folded into arrows, collapsed by multiply-and-add into one score, a meter with a crack in it.

Now step out of the music app and into a different aisle: score yourself against a job posting. Same meter, same arithmetic, just a new pair of lists. Before the code below fills it in: in this new aisle, which list plays the probe, you or the posting?


In [ ]:
qualities_job = ["python_skill", "night_shifts_ok", "years_experience"]
you = np.array([4.0, 1.0, 3.0])
posting = np.array([3.0, 0.0, 5.0])

fit_score = np.dot(you, posting)
print(f"fit score: {fit_score:.2f}")

# It's you doing the asking, same as the probe in the song example, and
# the multiply-and-add doesn't care which list you call the probe.
assert fit_score == np.dot(posting, you)


That's a standing invitation: anything you can describe as two matching lists, the meter can score. Edit the two arrays below to invent your own pair, then predict the sign of the score before you run it.


In [ ]:
# Invent your own pair here. Two short lists, same length, describing
# anything you can compare feature by feature.
mine_a = np.array([1.0, 2.0, 3.0])
mine_b = np.array([3.0, 2.0, 1.0])

my_score = np.dot(mine_a, mine_b)
print(f"your score: {my_score:.2f}")

assert mine_a.shape == mine_b.shape


## For the walk home

You built the meter, watched it read direction, and caught it faking a win on length alone.

For the walk home: what is the cheapest fix that keeps the direction and forgets the size?

Still up? Try this next: feed the meter two candidates where one is scaled 10x, 100x, and 1000x longer than the other, direction never moving, and watch how fast a longer arrow buries a better match on the scoreboard.

Next time the app hands you a stranger's song that is exactly right, you'll know the number that said ship it, and you'll know that number can be shouted over.
